!python -m pip install --upgrade pip
!pip install scanpy

In [1]:
import scanpy as sc
import pandas as pd
from recon.explore import Celltype
import numpy as np
import scanpy as sc  # single cell data
import pandas as pd  # data manipulation
import liana as li  # cell communication
import recon  # multilayer and perturbation prediction
import recon.data

In [2]:
path = "10M_PBMC_12donor_90cytokines_h5ad_20260203/20260203_Parse_10M_PBMC_cytokines.h5ad"
rna = sc.read_h5ad(path) # backed = "r"
print (rna.shape)

(9697974, 40352)


In [3]:
print (rna.obs["treatment"].unique())


['cytokine', 'PBS']
Categories (2, object): ['PBS', 'cytokine']

In [4]:
rna = rna[rna.obs["treatment"] == "PBS"]

In [5]:
print (rna)

View of AnnData object with n_obs × n_vars = 629701 × 40352
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'total_counts_MT', 'pct_counts_MT', 'log1p_total_counts_MT', 'donor', 'cytokine', 'treatment', 'cell_type'
    var: 'n_cells'


### Loading the GRNs

In [8]:
grn_path = "./GRMs_by_Pau_to_share/pbmc_hummus.csv"
grn = pd.read_csv(grn_path)
grn = grn.rename(columns={"score": "weight"})

In [9]:
grn

,source,cre,target,weight
0,TFAP2C,chr5-40410239-40410739,RPL37,0.000163
1,TFAP2C,chr5-40439075-40439575,RPL37,0.000163
2,TFAP2C,chr5-40486302-40486802,RPL37,0.000163
3,TFAP2C,chr5-40486873-40487373,RPL37,0.000163
4,TFAP2C,chr5-40502665-40503165,RPL37,0.000163
...,...,...,...,...
99998,VEZF1,chr20-59004826-59005326,GNAS,0.000022
99999,VEZF1,chr20-59016634-59017134,GNAS,0.000022
100000,VEZF1,chr20-59120435-59120935,GNAS,0.000022
100001,VEZF1,chr20-59161726-59162226,GNAS,0.000022


### Cell-Cell Communication

In [12]:
li.method.cellphonedb(rna, 
            # NOTE by default the resource uses HUMAN gene symbols
            resource_name="consensus", # mouseconsensus
            expr_prop=0.00,
            use_raw=False,
            groupby="cell_type",
            verbose=True, key_added='cpdb_res')
            

Using resource `consensus`.
Using `.X`!
/homes/shree/miniforge3/envs/recon/lib/python3.10/site-packages/anndata/_core/anndata.py:430: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
Make sure that normalized counts are passed!
/homes/shree/miniforge3/envs/recon/lib/python3.10/site-packages/liana/method/_pipe_utils/_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
['Y_RNA-18', 'Y_RNA-19', 'Y_RNA-21', 'Y_RNA-29', 'Y_RNA-33', 'Y_RNA-35', 'Y_RNA-43', 'Y_RNA-46', 'Y_RNA-62', 'Y_RNA-65', 'Y_RNA-66', 'Y_RNA-67', 'Y_RNA-68', 'Y_RNA-79', 'Y_RNA-85', 'Y_RNA-86', 'Y_RNA-93', 'Y_RNA-101', 'Y_RNA-121', 'Y_RNA-126', 'Y_RNA-127', 'Y_RNA-137', 'Y_RNA-163', 'Y_RNA-164', 'Y_RNA-176', 'Y_RNA-191', 'Y_RNA-196', 'Y_RNA-198', 'Y_RNA-215', 'Y_RNA-240', 'Y_RNA-247', 'Y_RNA-248', '

Generating ligand-receptor stats for 629701 samples and 1766 features


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [04:45<00:00,  3.50it/s]
/homes/shree/miniforge3/envs/recon/lib/python3.10/site-packages/liana/method/sc/_Method.py:319: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.


In [13]:
ccc_network = rna.uns["cpdb_res"].copy()
ccc_network = ccc_network[["ligand", "receptor", "lr_means", "source", "target"]]
ccc_network = ccc_network.rename(columns={
    "lr_means": "weight",
    "source": "celltype_source",
    "target": "celltype_target",
    "ligand": "source",
    "receptor": "target"
})
ccc_network = ccc_network[ccc_network['weight'] != 0]


### Save ccc_network to a file to save time

In [14]:
ccc_network.to_csv("ccc_network.csv", index = False)